# Lab 3 — Gradio Chat Interface for the Tiny Transformer



## 1. Load the saved checkpoint

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr

from pathlib import Path

CHECKPOINT_CANDIDATES = [
    r"C:\Users\LENOVO\OneDrive\Documents\GitHub\LLM\Large-Language-Models-Generative-AI\Lab-03-Tiny-Transformer\notebooks\tiny_transformer.pt",
    "tiny_transformer.pt",
    "Lab-03-Tiny-Transformer/notebooks/tiny_transformer.pt",
    "../Lab-03-Tiny-Transformer/notebooks/tiny_transformer.pt",
]
CHECKPOINT_PATH = next((p for p in CHECKPOINT_CANDIDATES if Path(p).exists()), None)
if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Could not find tiny_transformer.pt — run lab3_tiny_transformer.ipynb first so it "
        "saves the checkpoint, or point CHECKPOINT_PATH at its exact location."
    )
device = "cuda" if torch.cuda.is_available() else "cpu"


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim=64, num_heads=2,
                 num_layers=2, ff_dim=128):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(block_size, embed_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=0.0, batch_first=True, activation="gelu",
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(positions)
        mask = torch.triu(torch.ones(T, T, device=idx.device, dtype=torch.bool), diagonal=1)
        x = self.blocks(x, mask=mask)
        return self.lm_head(x)


checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
config = checkpoint["config"]
chars = checkpoint["chars"]
vocab_size = checkpoint["vocab_size"]

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(text):
    return [stoi[ch] for ch in text]


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


model = TinyTransformer(
    vocab_size,
    config["block_size"],
    embed_dim=config["embedding_dim"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    ff_dim=config["feedforward_dim"],
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print("Loaded checkpoint from tiny_transformer.pt")
print("Config:", config)
print("Parameter count:", f"{n_params:,}")


C:\Users\LENOVO\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded checkpoint from tiny_transformer.pt
Config: {'embedding_dim': 64, 'num_heads': 2, 'num_layers': 2, 'block_size': 64, 'feedforward_dim': 128}
Parameter count: 80,715


## 2. Generation function and the chat callback

Same generation loop as the main notebook, wrapped in the `chat_with_transformer(message, history)` function Gradio's `ChatInterface` calls on every message. Tested directly once below before wiring it into the UI.

In [2]:
@torch.no_grad()
def generate(prompt, max_new_chars=150, temperature=1.0):
    model.eval()
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_chars):
        idx_cond = ids[:, -config["block_size"]:]
        logits = model(idx_cond)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())


def chat_with_transformer(message, history):
    if not message or not message.strip():
        return "Please enter a message."
    supported = "".join(ch for ch in message if ch in stoi)
    if not supported:
        return "Please try text using characters in the training dataset."
    try:
        return generate(supported, max_new_chars=150)
    except Exception as e:
        return f"Text generation failed: {type(e).__name__}: {e}"


test_reply = chat_with_transformer("Once upon a time", [])
print('Direct function test — message: "Once upon a time"')
print("Reply:", test_reply)


Direct function test — message: "Once upon a time"
Reply: Once upon a time, Lily agal scom and says.

They smanky you fain shoder up. He tree, Lilye ran agenlptined they be witcriced the for the reeplanace to evered brean he


## 3. Build and launch the chat interface

`demo.launch()` normally opens an interactive chat window and blocks until stopped — that's what you'd call to actually use it. Since this notebook's committed output comes from a non-interactive batch run, the cell below instead launches it **non-blocking**, sends one real message through the running server via `gradio_client` (a genuine HTTP round trip, not just calling the Python function directly), and then closes the server — so the output below is proof the deployed app actually works, not just the underlying function.

**To chat with it interactively yourself:** run `demo.launch()` with no extra arguments in a new cell.

In [3]:
demo = gr.ChatInterface(
    fn=chat_with_transformer,
    title="My Tiny Transformer Chatbot",
    description="A character-level language model trained from scratch in this lab.",
)

import time

try:
    _, local_url, _ = demo.launch(
        prevent_thread_lock=True, quiet=True, show_error=True, inbrowser=False
    )
    time.sleep(2)
    print("Gradio app launched at:", local_url)

    from gradio_client import Client

    client = Client(local_url)
    result = client.predict("Once upon a time", api_name="/chat_with_transformer")
    print('Real HTTP round trip through the running Gradio server — message: "Once upon a time"')
    print("Server response:", result)
finally:
    demo.close()
    print("Server closed.")


Gradio app launched at: http://127.0.0.1:7860/
Loaded as API: http://127.0.0.1:7860/
Real HTTP round trip through the running Gradio server — message: "Once upon a time"
Server response: Once upon a time, there saw. The our maned," the pord whe so tharked. Apery."

He smer and paretned on. Samne do eatchest. She dary you lest any and the meaysiful por
Closing server running on port: 7860
Server closed.


In [4]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
